In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pandas_datareader import data as pdr
from datetime import datetime, timedelta
import statsmodels.api as sm
import os

In [ ]:
class Utils:
  def read_data(ticker, start, end, constrains):
    try:
      t = yf.Ticker(ticker)
      history = t.history(start=start, end=end, interval="1mo")[['Close', 'Volume']]
      cons_check = Utils.check_constrains(history["Close"].mean(), history["Volume"].mean(), constrains)
      close = history["Close"]


      if not cons_check:
        return False

      shares = t.info.get("sharesOutstanding")

      market_cap = close.values[-1] * shares

      annual_bs = t.balance_sheet
      annual_bv = annual_bs.loc['Stockholders Equity'].iloc[0]

      bm = annual_bv / market_cap


    except Exception as e:
      raise Exception(f"Error retrieving data for {ticker}: {e}")

    else:
      returns = close.pct_change().values
      momentum = 1
      for i in range(1, 12):
        momentum *= (1 + returns[-i])

      momentum -= 1
      return returns, market_cap, bm, momentum

  def check_constrains(close_mean, volume_mean, constrains):
    volume_mean = volume_mean / 1000000
    
    if constrains is None:
      return True

    if close_mean < constrains["min_price"] or volume_mean < constrains["min_volume"]:
      return False

    return True


In [ ]:
class DataReader:
  def __init__(self, tickers, start, end, constrains, reader_params):
    self.start = start
    self.end = end
    self.tickers = tickers
    self.reader_params = reader_params
    self.factors = self.read_factors(reader_params)
    self.constrains = constrains
    self.market_data = None


  def generate_data(self):
    data = self.read_market_data(
        self.tickers, self.start, 
        self.end, self.constrains
    )
    self.market_data, self.BM, self.m_caps, self.momentum = data

  def read_factors(self, reader_params):
    factors = pdr.DataReader(*self.reader_params)
    factor_data = factors[0]
    factor_data = factor_data[self.start:self.end]
    return factor_data


  def read_market_data(self, tickers, start, end, constrains):
    price_data = {}
    market_caps = {}
    momentum = {}
    BMs = {}


    for ticker in tickers:
      res = Utils.read_data(ticker, start, end, constrains)

      if not res:
        continue

      returns, market_cap, bm, mm = res
      price_data[ticker] = returns
      market_caps[ticker] = market_cap
      BMs[ticker] = bm
      momentum[ticker] = mm

    m_caps = pd.Series(market_caps)
    BM = pd.Series(BMs)
    mm = pd.Series(momentum)

    for k in price_data.keys():
      print(f"{k}: {len(price_data[k])}")

    m_data = pd.DataFrame(price_data).dropna()


    return m_data, BM, m_caps, mm

  @property
  def datastore(self):
    return {
      "BM": self.BM,
      "factors": self.factors,
      "market_data": self.market_data,
      "size": np.log(self.m_caps),
      "momentum": self.momentum
    } if self.market_data is not None else None



In [ ]:
class FeatureStore(DataReader):
  def __init__(self, tickers, start, end, cons, reader_params):
    super().__init__(tickers, start, end, cons, reader_params)

    self.generate_data()

    factors = self.datastore["factors"]
    market_data = self.datastore["market_data"]

    self.load_or_generate_data()
    

  def load_or_generate_data(self):
    returns_path = f"returns_{self.start}_{self.end}.pkl"
    factors_path = f"factors_{self.start}_{self.end}.pkl"
    signals_path = f"signals_{self.start}_{self.end}.pkl"

    if os.path.exists(returns_path) and os.path.exists(factors_path) and os.path.exists(signals_path):
      print("loading data")
      self.signals = pd.read_parquet(returns_path)
      self.returns = pd.read_parquet(factors_path)
      self.factors = pd.read_parquet(signals_path)

    else:
      print("generating data")
      self.signals = self.compute_signals()
      self.returns, self.factors = self.construct_factors()


  def compute_signals(self):
    omegas = {}
    raw_signals = {}

    for i in ['size', "BM", "momentum"]:
      x = self.datastore[i]
      coef = -1 if i == 'size' else 1
      raw_signals[i], sigma = self.compute_z_factor(x)
      raw_signals[i] *= coef
      omegas[i] = 1/sigma

    raw_signals_df = pd.DataFrame(raw_signals).T
    omegas = pd.Series(omegas)

    signals = raw_signals_df.mul(omegas, axis=0)

    return signals


  def compute_z_factor(self, x):
    mu = np.mean(x)
    sigma = np.std(x)

    z = (x - mu) / sigma

    return z, sigma

  def construct_factors(self):
    returns = self.datastore["market_data"]
    
    factors = self.datastore["factors"]
    diff = len(factors["RF"].values) - len(returns.values)
    rf = self.datastore["factors"]["RF"].values[diff:]
    factors.drop(["RF"], axis=1, inplace=True)

    for ticker in returns.columns:
      returns[ticker] -= rf
    
    
    return returns, factors


In [ ]:
class FactorModel:
  def __init__(self, X, returns):
    diff = (len(X)- len(returns))
    self.X = X[diff:]
    self.y = returns

  def fit(self):
    betas = {}
    for ticker in self.y.columns:
      x = sm.add_constant(self.X)
      y = self.y[ticker].values

      self.model = sm.OLS(y, x)
      results = self.model.fit()
      b = results.params

      betas[ticker] = b

    betas = pd.DataFrame(betas).drop(["const"], axis=0)
    return betas



In [ ]:
class FactorPremiaModel:
  def __init__(self):
    pass

  def fit(self):
    pass

In [ ]:
class HighRiskPortfolio():
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    self.model = FactorModel()
    self.premia_model = FactorModel()
    self.value_split()
    self.create_portfolios()
    self.construct_factors()
    self._init_model(self.model)
    self._init_factor_premia_model(self.premia_model)
    self.save_model = save_model
    self.model_path = model_path

  def _init_model(self):
    pass

  def _init_factor_premia_model(self):
    pass

  def _fit_factor_premia_model(self):
    pass
  
  def fit(self):
    if self.use_factor_premia_model:
      self._init_factor_premia_model()
      self._fit_factor_premia_model()

    self.model.fit()
    return self.model

  def _save_model(self):
    pass

  def build_portfolio(self):
    self._score()





In [ ]:
class PortfolioConstruction:
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    pass